# Active Share Analysis: Tuleva Täiendav Kogumisfond vs MSCI ACWI

**11 March 2026**

Look-through analysis of Tuleva Täiendav Kogumisfond (Additional Savings Fund) model portfolio holdings vs the MSCI ACWI benchmark.
Active share measures how much the fund's portfolio differs from the index.

**Data sources:**
- TKF model portfolio — [published PDF](https://tuleva.ee/wp-content/uploads/2026/01/Tuleva-Taiendav-Kogumisfond-mudelportfell-avalikustamiseks.pdf)
- ETF holdings (look-through) — downloaded from each ETF provider's website
- ACWI benchmark — SPDR MSCI ACWI ETF (SPYY) via ssga.com (daily holdings with ISINs)

In [1]:
import sys
import os
import io
import re
import time
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from bs4 import BeautifulSoup
from IPython.display import HTML, display

project_root = Path.cwd().parents[1]
sys.path.insert(0, str(project_root / 'common' / 'scripts'))
from generate_charts import setup_plot_style, TULEVA_BLUE, TULEVA_NAVY, TULEVA_MID_BLUE

setup_plot_style()

from dotenv import load_dotenv
load_dotenv(project_root / '.env')

# Brand colors
COLOR_OVERWEIGHT = TULEVA_NAVY
COLOR_UNDERWEIGHT = '#FF4800'
COLOR_NEUTRAL = '#B0B0B0'

# SPDR ACWI benchmark (daily holdings with ISINs)
SPDR_ACWI_URL = ('https://www.ssga.com/uk/en_gb/intermediary/etfs/library-content/'
                 'products/fund-data/etfs/emea/holdings-daily-emea-en-spyy-gy.xlsx')

# iShares config (for SAWD section comparison + ESG screening analysis)
ISHARES_AJAX = '1506575576011.ajax'
ISHARES_BASE = 'https://www.ishares.com/uk/individual/en/products'
ISHARES_UA = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)'

ISHARES_PRODUCTS = {
    'SAWD': {'id': 305419, 'slug': 'ishares-msci-world-esg-screened-ucits-etf-usd-acc-fund'},
    'SSAC': {'id': 251850, 'slug': 'ishares-msci-acwi-ucits-etf'},
    'SACW': {'id': 345063, 'slug': '345063'},
}

# Local holdings files
HOLDINGS_DIR = Path.cwd() / 'data' / 'tkf_holdings'

# EM countries (for separating DM/EM in ACWI)
EM_COUNTRIES = {
    'China', 'Taiwan', 'India', 'Korea (South)', 'Brazil', 'South Africa',
    'Mexico', 'Saudi Arabia', 'Thailand', 'Indonesia', 'Malaysia', 'Philippines',
    'Turkey', 'Poland', 'Chile', 'Qatar', 'United Arab Emirates', 'Kuwait',
    'Colombia', 'Peru', 'Czech Republic', 'Egypt', 'Greece', 'Hungary',
}

ISIN_RE = re.compile(r'^[A-Z]{2}[A-Z0-9]{9}[0-9]$')


def fetch_ishares_holdings(ticker):
    """Download full holdings CSV from iShares website."""
    product = ISHARES_PRODUCTS[ticker]
    url = (f'{ISHARES_BASE}/{product["id"]}/{product["slug"]}'
           f'/{ISHARES_AJAX}?fileType=csv&fileName={ticker}_holdings&dataType=fund')
    resp = requests.get(url, headers={'User-Agent': ISHARES_UA})
    resp.raise_for_status()
    lines = resp.text.splitlines()
    date_line = lines[0] if lines else ''
    holdings_date = date_line.split(',')[1].strip().strip('"') if ',' in date_line else None
    csv_start = next(i for i, l in enumerate(lines) if l.startswith('Ticker,'))
    df = pd.read_csv(io.StringIO('\n'.join(lines[csv_start:])), thousands=',')
    df['Weight (%)'] = pd.to_numeric(df['Weight (%)'], errors='coerce')
    return df, holdings_date


def fetch_spdr_acwi():
    """Download SPDR MSCI ACWI holdings (has ISINs for all stocks)."""
    resp = requests.get(SPDR_ACWI_URL, headers={'User-Agent': ISHARES_UA})
    resp.raise_for_status()
    df = pd.read_excel(io.BytesIO(resp.content), skiprows=5, engine='calamine')
    # Filter to valid equities (valid ISIN)
    df = df[df['ISIN'].astype(str).str.match(ISIN_RE, na=False)].copy()
    df['Percent of Fund'] = pd.to_numeric(df['Percent of Fund'], errors='coerce')
    return df


def normalize_name(name, keep_class=False):
    """Aggressively normalize stock names for cross-provider matching."""
    n = str(name).upper().strip()
    n = n.replace('.', ' ')
    n = n.replace("'", '')
    n = n.replace('-', ' ')
    n = n.replace('/', ' ')
    n = re.sub(r'^THE\s+', '', n)
    n = re.sub(r'\s+THE\s*$', '', n)
    n = re.sub(r'\s*\([A-Z]+\)\s*', ' ', n)
    n = re.sub(r'\s+(KRW|TWD|INR|CNY|HKD|ZAR|SAR|BRL|MXN|THB|IDR|MYR|PHP|PLN|CLP|QAR|AED|KWD|COP|PEN|CZK|EGP|HUF|TRY|USD|NPV)\s*[\d.]*', '', n)
    n = re.sub(r'\s+(?:SUBORDINATE VOTING|NON VOTING|PFD|PREF)\s*', ' ', n)
    if not keep_class:
        n = re.sub(r'\s+(?:CLASS [A-Z]\b|CL [A-Z]\b)\s*', ' ', n)
        n = re.sub(r'\s+CLA?\s*$', '', n)
    n = re.sub(r'\b([A-Z]) ([A-Z])\b', r'\1\2', n)
    n = re.sub(r'\b([A-Z]) ([A-Z])\b', r'\1\2', n)
    for _ in range(3):
        n = re.sub(r'\s+(AKTIENGESELLSCHAFT|INCORPORATED|CORPORATION|CORP|COMPANIES|COMPANY|COS|LIMITED|LTD|INC|PLC|CO|SA|AG|NV|SE|SPA|AB|ASA|OYJ|TBK|BHD|PCL|BERHAD|SH|ADS|REIT|DIVIDEND RIGHT CERT|PAR|N|SOCIETE ANONYME.*)\s*$', '', n)
    n = re.sub(r'\s+SPA\s*$', '', n)
    n = re.sub(r'\s+A/S\s*$', '', n)
    n = n.replace(' & ', ' AND ')
    n = re.sub(r'\s*(AND|&)\s*$', '', n)
    n = re.sub(r'\s+', ' ', n).strip()
    return n


def is_equity(row):
    return row.get('Asset Class', '') == 'Equity'


def scrollable_table(df, title='', max_height=400, fmt=None):
    """Render a DataFrame as a scrollable HTML table."""
    styled = df.style
    if fmt:
        styled = styled.format(fmt)
    styled = styled.set_table_styles([
        {'selector': 'th', 'props': [('position', 'sticky'), ('top', '0'),
                                      ('background', '#002F63'), ('color', 'white'),
                                      ('padding', '6px 10px'), ('font-size', '0.85em')]},
        {'selector': 'td', 'props': [('padding', '4px 10px'), ('font-size', '0.85em'),
                                      ('border-bottom', '1px solid #eee')]},
        {'selector': 'tr:hover td', 'props': [('background', '#f5f5f5')]},
    ])
    html = styled.to_html()
    title_html = f'<h4 style="margin: 15px 0 5px 0;">{title}</h4>' if title else ''
    footer = f'<p style="color: #666; font-size: 0.85em; margin: 3px 0 15px 0;">{len(df)} rows</p>'
    return HTML(f'{title_html}<div style="max-height: {max_height}px; overflow-y: auto; '
                f'border: 1px solid #ddd; margin: 5px 0;">{html}</div>{footer}')


def metric_box(label, value, note=''):
    """Render a key metric as styled HTML."""
    note_html = f'<p style="color:#666; font-size:0.85em; margin:3px 0 0 0;">{note}</p>' if note else ''
    return HTML(f'<div style="background:#f8f9fa; border-left:4px solid #002F63; padding:12px 16px; '
                f'margin:10px 0; border-radius:0 4px 4px 0;">'
                f'<span style="font-size:0.9em; color:#666;">{label}</span><br>'
                f'<span style="font-size:1.8em; font-weight:bold; color:#002F63;">{value}</span>'
                f'{note_html}</div>')


def diff_table(df, weight_col, bench_col, name_col='name', title='', n=20):
    """Render a scrollable table of weight differences."""
    t = df.sort_values('abs_diff', ascending=False).head(n).copy()
    t['Direction'] = t['diff'].apply(lambda x: 'OW' if x > 0 else 'UW')
    out = t[[name_col, weight_col, bench_col, 'diff', 'Direction']].copy()
    out.columns = ['Stock', 'Fund %', 'Bench %', 'Diff', '']
    out.index = range(1, len(out) + 1)
    return scrollable_table(out, title=title,
                           fmt={'Fund %': '{:.2%}', 'Bench %': '{:.2%}', 'Diff': '{:+.2%}'})


print('Setup complete')

Setup complete


In [2]:
# TKF model portfolio (from published PDF, Dec 2025)
model_portfolio = pd.DataFrame([
    {'name': 'Invesco MSCI EM Universal Screened UCITS ETF Acc',       'isin': 'IE00BMDBMY19', 'weight': 10.00, 'source': 'companiesmarketcap'},
    {'name': 'iShares Developed World Screened Index Fund',            'isin': 'IE00BFG1TM61', 'weight': 19.00, 'source': 'ishares_SAWD'},
    {'name': 'Xtrackers MSCI USA Screened UCITS ETF',                  'isin': 'IE00BJZ2DC62', 'weight': 17.40, 'source': 'xtrackers_usa'},
    {'name': 'Xtrackers MSCI Canada Screened UCITS ETF',               'isin': 'LU0476289540', 'weight':  1.60, 'source': 'xtrackers_canada'},
    {'name': 'ICAV Amundi MSCI USA Screened UCITS ETF',                'isin': 'IE000F60HVH9', 'weight': 18.50, 'source': 'amundi_usa'},
    {'name': 'Vanguard ESG North America All Cap UCITS ETF',           'isin': 'IE000O58J820', 'weight': 15.40, 'source': 'vanguard_na'},
    {'name': 'BNP Paribas Easy MSCI EUROPE MIN TE UCITS ETF',         'isin': 'LU1291099718', 'weight': 12.50, 'source': 'bnp_europe'},
    {'name': 'BNP Paribas Easy MSCI Pacific ex Japan Min TE UCITS ETF','isin': 'LU1291106356', 'weight':  1.90, 'source': 'bnp_pacific'},
    {'name': 'BNP Paribas Easy MSCI Japan Min TE UCITS ETF',          'isin': 'LU1291102447', 'weight':  3.70, 'source': 'bnp_japan'},
])

mp_display = model_portfolio[['name', 'isin', 'weight']].copy()
mp_display.index = range(1, len(mp_display) + 1)
display(scrollable_table(mp_display, title='TKF Model Portfolio', max_height=350,
                         fmt={'weight': '{:.1f}%'}))

,name,isin,weight
1,Invesco MSCI EM Universal Screened UCITS ETF Acc,IE00BMDBMY19,10.0%
2,iShares Developed World Screened Index Fund,IE00BFG1TM61,19.0%
3,Xtrackers MSCI USA Screened UCITS ETF,IE00BJZ2DC62,17.4%
4,Xtrackers MSCI Canada Screened UCITS ETF,LU0476289540,1.6%
5,ICAV Amundi MSCI USA Screened UCITS ETF,IE000F60HVH9,18.5%
6,Vanguard ESG North America All Cap UCITS ETF,IE000O58J820,15.4%
7,BNP Paribas Easy MSCI EUROPE MIN TE UCITS ETF,LU1291099718,12.5%
8,BNP Paribas Easy MSCI Pacific ex Japan Min TE UCITS ETF,LU1291106356,1.9%
9,BNP Paribas Easy MSCI Japan Min TE UCITS ETF,LU1291102447,3.7%


In [3]:
# Load ETF holdings from all sources
# Each loader returns a DataFrame with columns: name, isin, weight (fraction 0-1)
# iShares SAWD and Vanguard lack ISINs — we'll bridge them via SPDR later

all_etf_holdings = {}  # source_key → DataFrame
holdings_dates = {}


def load_xtrackers(filename):
    df = pd.read_excel(HOLDINGS_DIR / filename, skiprows=3,
                       names=['#', 'Name', 'ISIN', 'Country', 'Currency', 'Exchange',
                              'Type', 'Rating', 'Primary_Listing', 'Sector', 'Weight'])
    df = df.dropna(subset=['Name'])
    df['Weight'] = pd.to_numeric(df['Weight'], errors='coerce')
    return df[['Name', 'ISIN', 'Country', 'Sector', 'Weight']].rename(
        columns={'Name': 'name', 'ISIN': 'isin', 'Country': 'country', 'Sector': 'sector', 'Weight': 'weight'})


def load_amundi(filename):
    df = pd.read_excel(HOLDINGS_DIR / filename, engine='calamine', header=None, skiprows=20)
    df.columns = ['_skip', 'ISIN', 'Name', 'Asset_Class', 'Currency', 'Weight', 'Sector', 'Country'] + \
                 [f'_c{i}' for i in range(max(0, len(df.columns) - 8))]
    df = df.dropna(subset=['Name'])
    df['Weight'] = pd.to_numeric(df['Weight'], errors='coerce')
    return df[['Name', 'ISIN', 'Country', 'Sector', 'Weight']].rename(
        columns={'Name': 'name', 'ISIN': 'isin', 'Country': 'country', 'Sector': 'sector', 'Weight': 'weight'})


def load_vanguard(filename):
    df = pd.read_excel(HOLDINGS_DIR / filename, skiprows=6)
    df.columns = ['Ticker', 'Name', 'Weight', 'Sector', 'Region'] + list(df.columns[5:])
    df = df.dropna(subset=['Name'])
    df['Weight'] = df['Weight'].astype(str).str.rstrip('%').astype(float) / 100
    df['Country'] = df['Region'].map({'US': 'United States', 'CA': 'Canada'})
    return df[['Ticker', 'Name', 'Country', 'Sector', 'Weight']].rename(
        columns={'Ticker': 'ticker', 'Name': 'name', 'Country': 'country', 'Sector': 'sector', 'Weight': 'weight'})


def load_bnp(filename):
    df = pd.read_excel(HOLDINGS_DIR / filename, header=None, engine='calamine', skiprows=22)
    df.columns = ['_skip', 'Asset_Class', 'Name', 'ISIN', 'Weight'] + \
                 [f'_c{i}' for i in range(max(0, len(df.columns) - 5))]
    df = df.dropna(subset=['Name'])
    df['Weight'] = pd.to_numeric(df['Weight'], errors='coerce')
    df = df[df['Asset_Class'] == 'Equity']
    return df[['Name', 'ISIN', 'Weight']].rename(
        columns={'Name': 'name', 'ISIN': 'isin', 'Weight': 'weight'})


def load_invesco_from_web():
    """Fetch Invesco MSCI EM Universal Screened holdings from companiesmarketcap.com."""
    url = 'https://companiesmarketcap.com/gbp/invesco-msci-emerging-markets-universal-screened-ucits-etf/holdings/'
    r = requests.get(url, headers={'User-Agent': ISHARES_UA})
    soup = BeautifulSoup(r.text, 'html.parser')
    table = soup.find('table')
    rows = []
    for tr in table.find_all('tr')[1:]:
        cells = tr.find_all('td')
        if len(cells) >= 3:
            weight_text = cells[0].get_text(strip=True).rstrip('%')
            name = cells[1].get_text(strip=True)
            isin = cells[2].get_text(strip=True)
            try:
                weight = float(weight_text) / 100
            except ValueError:
                continue
            rows.append({'name': name, 'isin': isin, 'weight': weight})
    return pd.DataFrame(rows)


# --- Load all holdings ---

# 1. Xtrackers USA
print('Loading Xtrackers USA...', end=' ')
all_etf_holdings['xtrackers_usa'] = load_xtrackers('Constituent_IE00BJZ2DC62.xlsx')
holdings_dates['xtrackers_usa'] = 'Mar 2026'
print(f'{len(all_etf_holdings["xtrackers_usa"])} holdings')

# 2. Xtrackers Canada
print('Loading Xtrackers Canada...', end=' ')
all_etf_holdings['xtrackers_canada'] = load_xtrackers('Constituent_LU0476289540.xlsx')
holdings_dates['xtrackers_canada'] = 'Mar 2026'
print(f'{len(all_etf_holdings["xtrackers_canada"])} holdings')

# 3. Amundi USA
print('Loading Amundi USA...', end=' ')
all_etf_holdings['amundi_usa'] = load_amundi(
    'Fund Holdings_Amundi MSCI USA Screened UCITS ETF Acc_IE000F60HVH9_09_03_2026.xlsx')
holdings_dates['amundi_usa'] = '9 Mar 2026'
print(f'{len(all_etf_holdings["amundi_usa"])} holdings')

# 4. Vanguard NA (stale: 31 Jan 2026)
print('Loading Vanguard NA...', end=' ')
all_etf_holdings['vanguard_na'] = load_vanguard(
    'Holdings details - Vanguard ESG North America All Cap UCITS ETF (USD) Accumulating - 3_11_2026.xlsx')
holdings_dates['vanguard_na'] = '31 Jan 2026 (stale — estimated effect on active share < 1.5%)'
print(f'{len(all_etf_holdings["vanguard_na"])} holdings')

# 5. BNP Europe
print('Loading BNP Europe...', end=' ')
all_etf_holdings['bnp_europe'] = load_bnp('Fund_Holdings_INV_LU1291099718_en.xlsx')
holdings_dates['bnp_europe'] = '6 Mar 2026'
print(f'{len(all_etf_holdings["bnp_europe"])} holdings')

# 6. BNP Pacific ex Japan
print('Loading BNP Pacific ex Japan...', end=' ')
all_etf_holdings['bnp_pacific'] = load_bnp('Fund_Holdings_INV_LU1291106356_en.xlsx')
holdings_dates['bnp_pacific'] = '6 Mar 2026'
print(f'{len(all_etf_holdings["bnp_pacific"])} holdings')

# 7. BNP Japan
print('Loading BNP Japan...', end=' ')
all_etf_holdings['bnp_japan'] = load_bnp('Fund_Holdings_INV_LU1291102447_en.xlsx')
holdings_dates['bnp_japan'] = '6 Mar 2026'
print(f'{len(all_etf_holdings["bnp_japan"])} holdings')

# 8. iShares Developed World Screened (fetched live — no ISIN column, uses Ticker)
print('Fetching iShares SAWD...', end=' ')
sawd_raw, sawd_date = fetch_ishares_holdings('SAWD')
sawd_eq = sawd_raw[sawd_raw.apply(is_equity, axis=1)].copy()
sawd_eq['weight'] = sawd_eq['Weight (%)'] / 100
sawd_eq = sawd_eq.rename(columns={'Name': 'name', 'Location': 'country', 'Sector': 'sector', 'Ticker': 'ticker'})
all_etf_holdings['ishares_SAWD'] = sawd_eq[['ticker', 'name', 'country', 'sector', 'weight']]
holdings_dates['ishares_SAWD'] = sawd_date
print(f'{len(sawd_eq)} equities (as of {sawd_date})')
time.sleep(0.5)

# 9. Invesco EM (fetched from companiesmarketcap)
print('Fetching Invesco EM from companiesmarketcap.com...', end=' ')
all_etf_holdings['companiesmarketcap'] = load_invesco_from_web()
holdings_dates['companiesmarketcap'] = 'Mar 2026'
print(f'{len(all_etf_holdings["companiesmarketcap"])} holdings')

# 10. SPDR MSCI ACWI (benchmark — has ISINs for all stocks)
print('Fetching SPDR MSCI ACWI (SPYY)...', end=' ')
spdr_acwi = fetch_spdr_acwi()
print(f'{len(spdr_acwi)} equities')

# Build ISIN→name/country lookup from SPDR (used as ticker→ISIN bridge for SAWD/Vanguard)
spdr_by_isin = (spdr_acwi.groupby('ISIN')
    .agg({'Security Name': 'first', 'Trade Country Name': 'first', 'Percent of Fund': 'sum'})
    .to_dict('index'))
spdr_by_name = {normalize_name(r['Security Name']): r['ISIN']
                for _, r in spdr_acwi.iterrows() if pd.notna(r['Security Name'])}

# Summary
print(f'\n{"Source":<25s} {"Holdings":>8s}  {"Weight sum":>10s}  Date')
print('-' * 70)
for key, df in all_etf_holdings.items():
    print(f'{key:<25s} {len(df):>8d}  {df["weight"].sum():>10.4f}  {holdings_dates[key]}')
print(f'{"spdr_acwi (benchmark)":<25s} {len(spdr_acwi):>8d}  {spdr_acwi["Percent of Fund"].sum():>10.4f}')

Loading Xtrackers USA... 486 holdings
Loading Xtrackers Canada... 76 holdings
Loading Amundi USA... 483 holdings
Loading Vanguard NA... 

1415 holdings
Loading BNP Europe... 362 holdings
Loading BNP Pacific ex Japan... 89 holdings
Loading BNP Japan... 158 holdings
Fetching iShares SAWD... 

1201 equities (as of 10/Mar/2026)


Fetching Invesco EM from companiesmarketcap.com... 

472 holdings
Fetching SPDR MSCI ACWI (SPYY)... 

2284 equities

Source                    Holdings  Weight sum  Date
----------------------------------------------------------------------
xtrackers_usa                  486      1.0000  Mar 2026
xtrackers_canada                76      1.0000  Mar 2026
amundi_usa                     483      1.0005  9 Mar 2026
vanguard_na                   1415      0.9933  31 Jan 2026 (stale — estimated effect on active share < 1.5%)
bnp_europe                     362      0.9954  6 Mar 2026
bnp_pacific                     89      0.9921  6 Mar 2026
bnp_japan                      158      0.9583  6 Mar 2026
ishares_SAWD                  1201      0.9978  10/Mar/2026
companiesmarketcap             472      0.9298  Mar 2026
spdr_acwi (benchmark)         2284     99.9989


In [4]:
# Build look-through TKF portfolio using ISIN as the primary stock identifier
# For SAWD and Vanguard (no ISINs), bridge via SPDR name matching
# Unmatched stocks kept with ticker-based IDs (they're small caps not in ACWI)

# Pre-build SPDR lookups for bridging — two passes: with class info, then without
spdr_name_class = {}   # normalized name WITH class → ISIN (precise)
spdr_name_to_isin = {} # normalized name WITHOUT class → ISIN (fallback)
spdr_prefix_country = {}
spdr_prefix_only = {}
for _, r in spdr_acwi.iterrows():
    norm_cls = normalize_name(r['Security Name'], keep_class=True)
    norm = normalize_name(r['Security Name'], keep_class=False)
    country = r['Trade Country Name']
    spdr_name_class[norm_cls] = r['ISIN']
    if norm not in spdr_name_to_isin:
        spdr_name_to_isin[norm] = r['ISIN']
    for plen in [15, 10, 8, 6, 4]:
        if len(norm) >= plen:
            key = (norm[:plen], country)
            if key not in spdr_prefix_country:
                spdr_prefix_country[key] = r['ISIN']
            if plen >= 10 and norm[:plen] not in spdr_prefix_only:
                spdr_prefix_only[norm[:plen]] = r['ISIN']

def bridge_to_isin(ticker, name, country):
    """Try to find ISIN for a stock without one, using SPDR as bridge."""
    # 1. Try with class info preserved (distinguishes Class A vs C)
    norm_cls = normalize_name(name, keep_class=True)
    if norm_cls in spdr_name_class:
        return spdr_name_class[norm_cls]
    # 2. Try without class info
    norm = normalize_name(name, keep_class=False)
    if norm in spdr_name_to_isin:
        return spdr_name_to_isin[norm]
    # 3. Prefix + country (try longer first, then shorter)
    if country:
        for plen in [15, 10, 8, 6, 4]:
            if len(norm) >= plen:
                key = (norm[:plen], country)
                if key in spdr_prefix_country:
                    return spdr_prefix_country[key]
    # 4. Prefix without country (safe at 10+ chars)
    for plen in [15, 10]:
        if len(norm) >= plen and norm[:plen] in spdr_prefix_only:
            return spdr_prefix_only[norm[:plen]]
    return None

# Build look-through portfolio
all_stocks = []
bridge_stats = {'bridged': 0, 'unmatched': 0, 'unmatched_weight': 0}

for _, mp_row in model_portfolio.iterrows():
    source = mp_row['source']
    fund_weight = mp_row['weight'] / 100.0
    df_h = all_etf_holdings[source]
    source_bridged = 0

    for _, h in df_h.iterrows():
        isin = h.get('isin', '')
        if pd.notna(isin) and ISIN_RE.match(str(isin)):
            stock_isin = str(isin)
        else:
            # No ISIN — bridge via SPDR
            bridged = bridge_to_isin(h.get('ticker', ''), h['name'], h.get('country', ''))
            if bridged:
                stock_isin = bridged
                source_bridged += 1
                bridge_stats['bridged'] += 1
            else:
                # Keep with synthetic ID — these are small caps not in ACWI
                ticker = h.get('ticker', h['name'][:20])
                stock_isin = f'_UNBRIDGED_{ticker}'
                bridge_stats['unmatched'] += 1
                bridge_stats['unmatched_weight'] += fund_weight * h['weight'] * 100

        all_stocks.append({
            'isin': stock_isin,
            'name': h['name'],
            'weight': fund_weight * h['weight'] * 100,
            'sector': h.get('sector', ''),
            'country': h.get('country', ''),
            'source_etf': source,
        })

    bridged_note = f' ({source_bridged} bridged via SPDR)' if source_bridged else ''
    print(f'{mp_row["name"][:50]:50s}  {mp_row["weight"]:5.1f}% × {len(df_h):>5d} stocks{bridged_note}')

tkf_raw = pd.DataFrame(all_stocks)

# Merge preferred → common share classes
SHARE_CLASS_MERGE = {
    'KR7005931001': 'KR7005930003',  # Samsung Electronics pref → common
}
tkf_raw['isin'] = tkf_raw['isin'].replace(SHARE_CLASS_MERGE)

# Aggregate by ISIN (same stock from multiple ETFs)
tkf_portfolio = (tkf_raw
    .groupby('isin')
    .agg({'name': 'first', 'weight': 'sum', 'sector': 'first', 'country': 'first'})
    .reset_index()
)

tkf_portfolio = tkf_portfolio.sort_values('weight', ascending=False).reset_index(drop=True)

print(f'\nBridging: {bridge_stats["bridged"]} tickers→ISINs via SPDR, '
      f'{bridge_stats["unmatched"]} unmatched ({bridge_stats["unmatched_weight"]:.2f}% weight — mostly small caps not in ACWI)')
print(f'TKF look-through: {len(tkf_portfolio):,} unique stocks ({tkf_portfolio["weight"].sum():.2f}% total)')
print(f'\nTop 20 holdings:')
display(tkf_portfolio.head(20)[['isin', 'name', 'weight', 'sector', 'country']]
        .style.format({'weight': '{:.4f}%'}))

Invesco MSCI EM Universal Screened UCITS ETF Acc     10.0% ×   472 stocks
iShares Developed World Screened Index Fund          19.0% ×  1201 stocks (1162 bridged via SPDR)
Xtrackers MSCI USA Screened UCITS ETF                17.4% ×   486 stocks
Xtrackers MSCI Canada Screened UCITS ETF              1.6% ×    76 stocks (1 bridged via SPDR)
ICAV Amundi MSCI USA Screened UCITS ETF              18.5% ×   483 stocks


Vanguard ESG North America All Cap UCITS ETF         15.4% ×  1415 stocks (641 bridged via SPDR)
BNP Paribas Easy MSCI EUROPE MIN TE UCITS ETF        12.5% ×   362 stocks
BNP Paribas Easy MSCI Pacific ex Japan Min TE UCIT    1.9% ×    89 stocks
BNP Paribas Easy MSCI Japan Min TE UCITS ETF          3.7% ×   158 stocks

Bridging: 1804 tickers→ISINs via SPDR, 819 unmatched (1.26% weight — mostly small caps not in ACWI)
TKF look-through: 2,527 unique stocks (98.94% total)

Top 20 holdings:


,isin,name,weight,sector,country
0,US67066G1040,NVIDIA CORP,5.2793%,Information Technology,United States
1,US0378331005,APPLE INC,4.5184%,Information Technology,United States
2,US5949181045,MICROSOFT CORP,3.4989%,Information Technology,United States
3,US02079K3059,ALPHABET INC CLASS A,2.6042%,Communication,United States
4,US0231351067,AMAZON COM INC,2.5028%,Consumer Discretionary,United States
5,US11135F1012,BROADCOM INC,1.8343%,Information Technology,United States
6,US30303M1027,META PLATFORMS INC CLASS A,1.7171%,Communication,United States
7,US88160R1014,TESLA INC,1.3599%,Consumer Discretionary,United States
8,US02079K1079,ALPHABET INC CLASS C,1.3551%,Communication,United States
9,US5324571083,ELI LILLY,0.9654%,Health Care,United States


In [5]:
# ACWI benchmark — SPDR MSCI ACWI (already fetched in cell above)
# No sub-ETF decomposition needed — SPDR holds all equities directly with ISINs

# Merge preferred → common share classes (same as TKF)
spdr_acwi['ISIN'] = spdr_acwi['ISIN'].replace(SHARE_CLASS_MERGE)

acwi_portfolio = (spdr_acwi
    .groupby('ISIN')
    .agg({'Security Name': 'first', 'Percent of Fund': 'sum',
          'Sector Classification': 'first', 'Trade Country Name': 'first'})
    .reset_index()
    .rename(columns={
        'ISIN': 'isin', 'Security Name': 'name', 'Percent of Fund': 'weight',
        'Sector Classification': 'sector', 'Trade Country Name': 'country',
    })
)

# Normalize to 100%
total_acwi = acwi_portfolio['weight'].sum()
acwi_portfolio['weight'] = acwi_portfolio['weight'] / total_acwi * 100
acwi_portfolio = acwi_portfolio.sort_values('weight', ascending=False).reset_index(drop=True)

print(f'ACWI benchmark (SPDR SPYY): {len(acwi_portfolio):,} unique stocks')
print(f'\nTop 10:')
display(acwi_portfolio.head(10)[['isin', 'name', 'weight', 'sector', 'country']]
        .style.format({'weight': '{:.4f}%'}))

ACWI benchmark (SPDR SPYY): 2,282 unique stocks

Top 10:


,isin,name,weight,sector,country
0,US67066G1040,NVIDIA Corporation,4.8236%,Information Technology,United States
1,US0378331005,Apple Inc.,4.0843%,Information Technology,United States
2,US5949181045,Microsoft Corporation,3.0372%,Information Technology,United States
3,US0231351067,Amazon.com Inc.,2.1670%,Consumer Discretionary,United States
4,US02079K3059,Alphabet Inc. Class A,1.9019%,Communication Services,United States
5,US11135F1012,Broadcom Inc.,1.6331%,Information Technology,United States
6,US02079K1079,Alphabet Inc. Class C,1.6243%,Communication Services,United States
7,TW0002330008,Taiwan Semiconductor Manufacturing Co. Ltd.,1.6067%,Information Technology,Taiwan
8,US30303M1027,Meta Platforms Inc Class A,1.5195%,Communication Services,United States
9,US88160R1014,Tesla Inc.,1.2300%,Consumer Discretionary,United States


In [6]:
# Compute active share: ½ × Σ|w_fund − w_benchmark|
# Direct ISIN join — no reconciliation layer needed

tkf_isins = set(tkf_portfolio['isin'])
acwi_isins = set(acwi_portfolio['isin'])

merged = pd.merge(
    tkf_portfolio[['isin', 'name', 'weight', 'sector', 'country']],
    acwi_portfolio[['isin', 'weight']].rename(columns={'weight': 'weight_acwi'}),
    on='isin', how='outer',
)

# Fill names from ACWI for stocks only in benchmark
acwi_only_mask = merged['name'].isna()
if acwi_only_mask.any():
    acwi_lookup = acwi_portfolio.set_index('isin')
    for idx in merged[acwi_only_mask].index:
        isin = merged.loc[idx, 'isin']
        if isin in acwi_lookup.index:
            merged.loc[idx, 'name'] = acwi_lookup.loc[isin, 'name']
            merged.loc[idx, 'sector'] = acwi_lookup.loc[isin, 'sector']
            merged.loc[idx, 'country'] = acwi_lookup.loc[isin, 'country']

merged['weight'] = merged['weight'].fillna(0)
merged['weight_acwi'] = merged['weight_acwi'].fillna(0)
merged['in_tkf'] = merged['isin'].isin(tkf_isins)
merged['in_acwi'] = merged['isin'].isin(acwi_isins)
merged['abs_diff'] = (merged['weight'] - merged['weight_acwi']).abs()
merged['diff'] = merged['weight'] - merged['weight_acwi']

active_share = merged['abs_diff'].sum() / 2
overlap = len(merged[merged['in_tkf'] & merged['in_acwi']])
tkf_only = merged[merged['in_tkf'] & ~merged['in_acwi']]
acwi_only = merged[~merged['in_tkf'] & merged['in_acwi']]

# Decomposition
weight_excluded = acwi_only['weight_acwi'].sum()
weight_added = tkf_only['weight'].sum()
overlapping = merged[merged['in_tkf'] & merged['in_acwi']]
weight_reduced = overlapping.loc[overlapping['diff'] < 0, 'diff'].abs().sum()

display(metric_box('Active Share — TKF vs MSCI ACWI', f'{active_share:.2f}%',
                   f'½ × Σ|w_fund − w_bench| · {len(tkf_portfolio):,} TKF stocks vs {len(acwi_portfolio):,} ACWI stocks · '
                   f'{overlap:,} overlapping · {len(tkf_only):,} TKF-only · {len(acwi_only):,} ACWI-only'))

decomp = pd.DataFrame([
    {'Component': 'Weight excluded (ACWI stocks not in TKF)', 'Weight %': weight_excluded, 'Stocks': len(acwi_only)},
    {'Component': 'Weight added (TKF stocks not in ACWI)', 'Weight %': weight_added, 'Stocks': len(tkf_only)},
    {'Component': 'Weight reduced (overlapping stocks underweight in TKF)', 'Weight %': weight_reduced,
     'Stocks': int((overlapping['diff'] < 0).sum())},
]).set_index('Component')
display(scrollable_table(decomp, title='Active share decomposition', max_height=200,
                         fmt={'Weight %': '{:.2f}'}))

# --- Weight excluded: ACWI stocks not in TKF ---
excluded_df = acwi_only.sort_values('weight_acwi', ascending=False).copy()
excl_out = excluded_df[['name', 'weight_acwi', 'country']].copy()
excl_out.columns = ['Stock', 'ACWI %', 'Country']
excl_out.index = range(1, len(excl_out) + 1)
display(scrollable_table(excl_out, title=f'Weight excluded — {len(acwi_only)} ACWI stocks not in TKF ({weight_excluded:.2f}%)',
                         max_height=350, fmt={'ACWI %': '{:.2f}'}))

# --- Weight added: TKF stocks not in ACWI ---
added_df = tkf_only.sort_values('weight', ascending=False).copy()
add_out = added_df[['name', 'weight', 'country']].copy()
add_out.columns = ['Stock', 'TKF %', 'Country']
add_out.index = range(1, len(add_out) + 1)
display(scrollable_table(add_out, title=f'Weight added — {len(tkf_only)} TKF stocks not in ACWI ({weight_added:.2f}%)',
                         max_height=350, fmt={'TKF %': '{:.2f}'}))

# --- Weight reduced: overlapping stocks underweight in TKF ---
reduced_df = overlapping[overlapping['diff'] < 0].sort_values('diff').copy()
red_out = reduced_df[['name', 'weight', 'weight_acwi', 'diff', 'country']].copy()
red_out.columns = ['Stock', 'TKF %', 'ACWI %', 'Diff', 'Country']
red_out.index = range(1, len(red_out) + 1)
display(scrollable_table(red_out, title=f'Weight reduced — {len(reduced_df)} stocks underweight in TKF ({weight_reduced:.2f}%)',
                         max_height=350, fmt={'TKF %': '{:.2f}', 'ACWI %': '{:.2f}', 'Diff': '{:+.2f}'}))

,Weight %,Stocks
Component,,
Weight excluded (ACWI stocks not in TKF),5.45,608
Weight added (TKF stocks not in ACWI),1.54,853
Weight reduced (overlapping stocks underweight in TKF),10.36,564


,Stock,ACWI %,Country
1,Chevron Corporation,0.39,United States
2,Philip Morris International Inc.,0.28,United States
3,Boeing Company,0.18,United States
4,Honeywell International Inc.,0.16,United States
5,ConocoPhillips,0.16,United States
6,Lockheed Martin Corporation,0.14,United States
7,British American Tobacco p.l.c.,0.13,United Kingdom
8,Altria Group Inc.,0.12,United States
9,Duke Energy Corporation,0.10,United States
10,Canadian Natural Resources Limited,0.10,Canada


,Stock,TKF %,Country
1,EMINI S&P500 ESG 03/26 CME,0.03,United States
2,LAIR LIQUIDE SOCIETE ANONYME POUR,0.03,France
3,GLAXOSMITHKLINE,0.03,United Kingdom
4,BOUBYAN BANK K.S.C KWd100,0.02,
5,CRH PLC,0.02,United States
6,Sandisk Corp/DE,0.02,United States
7,CHINA MOLYBDENUM CO LTD-A CNY 0.2000,0.02,
8,MUENCHENER RUECKVERSICHERUNGS-GESE,0.02,Germany
9,BAWAG GROUP AG,0.02,
10,PTT PCL-NVDR THB1,0.02,


,Stock,TKF %,ACWI %,Diff,Country
1,TAIWAN SEMICONDUCTOR MANUFAC TWD10,0.57,1.61,-1.03,
2,SAMSUNG ELECTRONICS-PREF KRW5000 PFD,0.08,0.76,-0.68,
3,BERKSHIRE HATHAWAY INC CLASS B,0.39,0.71,-0.32,United States
4,Procter & Gamble Co/The,0.10,0.38,-0.28,United States
5,ALPHABET INC CLASS C,1.36,1.62,-0.27,United States
6,Coca-Cola Co/The,0.08,0.33,-0.25,United States
7,RTX CORP,0.10,0.28,-0.18,United States
8,PepsiCo Inc,0.06,0.23,-0.17,United States
9,TOYOTA MOTOR CORP,0.05,0.23,-0.17,Japan
10,TENCENT HOLDINGS LTD HKD0.00002,0.37,0.50,-0.13,


## Sources of active share

### ESG screening

All ETFs in the TKF model portfolio apply some form of ESG screening. We measure the baseline ESG effect by comparing iShares MSCI ACWI (SSAC, unscreened) with iShares MSCI ACWI ESG Screened — same universe, differing only in exclusions. The excluded stocks' total weight in ACWI gives us the ESG screening baseline.

In [7]:
# ESG screening impact: ACWI (unscreened) vs iShares MSCI ACWI ESG Screened
# Uses iShares Ticker|Location matching (both from same provider, so tickers are consistent)

print('Fetching iShares SSAC (ACWI unscreened)...', end=' ')
ssac_raw, ssac_date = fetch_ishares_holdings('SSAC')
ssac_eq = ssac_raw[ssac_raw.apply(is_equity, axis=1)].copy()
ssac_eq['stock_id'] = ssac_eq.apply(lambda r: f"{r['Ticker']}|{r['Location']}", axis=1)

# SSAC has sub-ETFs (NDIA, 4BRZ, CNYA, IKSA) — for this comparison we keep them as-is
# since we're just checking which stocks are missing, not computing weights
print(f'{len(ssac_eq)} equities (as of {ssac_date})')

print('Fetching iShares SACW (ACWI ESG Screened)...', end=' ')
sacw_raw, sacw_date = fetch_ishares_holdings('SACW')
sacw_eq = sacw_raw[sacw_raw.apply(is_equity, axis=1)].copy()
sacw_eq['stock_id'] = sacw_eq.apply(lambda r: f"{r['Ticker']}|{r['Location']}", axis=1)
print(f'{len(sacw_eq)} equities (as of {sacw_date})')

# Normalize SSAC weights
ssac_eq['weight_norm'] = ssac_eq['Weight (%)'] / ssac_eq['Weight (%)'].sum()

# Stocks screened out
ssac_ids = set(ssac_eq['stock_id'])
sacw_ids = set(sacw_eq['stock_id'])
screened_out = ssac_ids - sacw_ids
screened_out_df = ssac_eq[ssac_eq['stock_id'].isin(screened_out)].copy()
screened_out_weight = screened_out_df['weight_norm'].sum()

display(metric_box('ESG Screening Baseline',
                   f'{screened_out_weight:.2%} of ACWI weight excluded',
                   f'{len(screened_out)} stocks removed by MSCI ESG screening · '
                   f'ACWI ({len(ssac_eq)} stocks) vs ACWI Screened ({len(sacw_eq)} stocks) · '
                   f'TKF active share of {active_share:.2f}% is ~{active_share / (screened_out_weight*100):.1f}× this baseline'))

# Table of excluded stocks
excluded = screened_out_df.sort_values('weight_norm', ascending=False).copy()
excluded_display = excluded[['Name', 'weight_norm']].copy()
excluded_display.columns = ['Stock', 'ACWI Weight %']
excluded_display.index = range(1, len(excluded_display) + 1)
display(scrollable_table(excluded_display,
                         title='ESG-excluded stocks (in ACWI but not in ACWI Screened)',
                         fmt={'ACWI Weight %': '{:.2%}'}))

Fetching iShares SSAC (ACWI unscreened)... 

1728 equities (as of 10/Mar/2026)
Fetching iShares SACW (ACWI ESG Screened)... 

1741 equities (as of 10/Mar/2026)


,Stock,ACWI Weight %
1,ISHARES MSCI INDIA UCITS ETF,1.51%
2,ISHARES MSCI BRAZIL UCITS ET USDHA,0.60%
3,ISH MSCI CHINA A ETF USD ACC,0.44%
4,PROCTER & GAMBLE,0.38%
5,CHEVRON CORP,0.38%
6,ISHARES MSCI SAUDI ARABIA CAPPED,0.34%
7,COCA-COLA,0.34%
8,RTX CORP,0.29%
9,NESTLE LTD,0.28%
10,PHILIP MORRIS INTERNATIONAL INC,0.28%


### US equities: Amundi + Xtrackers vs iShares

The TKF model portfolio holds US equities through three ESG-screened ETFs: Xtrackers MSCI USA Screened (17.4%), Amundi MSCI USA Screened (18.5%), and part of iShares MSCI World Screened (SAWD, 19%). All three track variants of MSCI USA ESG Screened — how much additional active share do Amundi and Xtrackers add beyond the iShares baseline?

We compare a 50/50 blend of Amundi + Xtrackers vs the US portion of SAWD. The result isolates provider-level differences (different share classes, slightly different screening dates/criteria) — not the ESG screening effect itself, which was measured above.

In [8]:
# US equities: Amundi + Xtrackers (50/50) vs SAWD US portion
# Bridge SAWD tickers to ISINs via SPDR, then join on ISIN

# Bridge SAWD to ISINs
sawd_with_isin = all_etf_holdings['ishares_SAWD'].copy()
sawd_with_isin['isin'] = sawd_with_isin.apply(
    lambda r: bridge_to_isin(r.get('ticker', ''), r['name'], r.get('country', '')), axis=1)

sawd_us = sawd_with_isin[sawd_with_isin['country'] == 'United States'].copy()
sawd_us_matched = sawd_us[sawd_us['isin'].notna()]
sawd_us_total = sawd_us_matched['weight'].sum()
sawd_us_matched = sawd_us_matched.copy()
sawd_us_matched['weight'] = sawd_us_matched['weight'] / sawd_us_total

print(f'SAWD US: {len(sawd_us)} stocks, {len(sawd_us_matched)} with ISINs ({len(sawd_us) - len(sawd_us_matched)} unbridged)')

# Blend Xtrackers + Amundi 50/50
xt_usa_h = all_etf_holdings['xtrackers_usa'].copy()
am_usa_h = all_etf_holdings['amundi_usa'].copy()
xt_usa_h['weight_blend'] = xt_usa_h['weight'] * 0.5
am_usa_h['weight_blend'] = am_usa_h['weight'] * 0.5

blend = pd.concat([xt_usa_h[['isin', 'name', 'weight_blend']], am_usa_h[['isin', 'name', 'weight_blend']]])
blend = blend.groupby('isin').agg({'name': 'first', 'weight_blend': 'sum'}).reset_index()
blend['weight'] = blend['weight_blend'] / blend['weight_blend'].sum()

merged_us = pd.merge(
    blend[['isin', 'name', 'weight']],
    sawd_us_matched[['isin', 'weight']].rename(columns={'weight': 'weight_sawd'}),
    on='isin', how='outer',
)
merged_us['weight'] = merged_us['weight'].fillna(0)
merged_us['weight_sawd'] = merged_us['weight_sawd'].fillna(0)
merged_us['abs_diff'] = (merged_us['weight'] - merged_us['weight_sawd']).abs()
merged_us['diff'] = merged_us['weight'] - merged_us['weight_sawd']
merged_us['name'] = merged_us['name'].fillna('')
for idx, r in merged_us[merged_us['name'] == ''].iterrows():
    match = sawd_us_matched[sawd_us_matched['isin'] == r['isin']]
    if len(match):
        merged_us.loc[idx, 'name'] = match.iloc[0]['name']

us_active_share = merged_us['abs_diff'].sum() / 2
overlap_n = len(set(blend['isin']) & set(sawd_us_matched['isin']))

display(metric_box('US: Amundi + Xtrackers vs iShares SAWD', f'{us_active_share:.2%}',
                   f'{len(blend)} blend stocks vs {len(sawd_us_matched)} SAWD US stocks · {overlap_n} overlapping · '
                   f'Small — same MSCI USA ESG Screened index family'))
display(diff_table(merged_us, 'weight', 'weight_sawd', title='Top weight differences (US)'))

SAWD US: 486 stocks, 482 with ISINs (4 unbridged)


,Stock,Fund %,Bench %,Diff,
1,BERKSHIRE HATHAWAY INC CLASS B,0.63%,1.24%,-0.62%,UW
2,RTX CORP,0.26%,0.00%,+0.26%,OW
3,GE VERNOVA INC,0.21%,0.42%,-0.21%,UW
4,YUM BRANDS INC,0.08%,0.00%,+0.08%,OW
5,NVIDIA CORP,8.16%,8.24%,-0.08%,UW
6,MICROSOFT CORP,5.31%,5.24%,+0.07%,OW
7,L3HARRIS TECHNOLOGIES INC,0.07%,0.00%,+0.07%,OW
8,PHILLIPS,0.06%,0.00%,+0.06%,OW
9,NORFOLK SOUTHERN CORP,0.06%,0.11%,-0.05%,UW
10,BROADCOM INC,2.86%,2.83%,+0.04%,OW


### EM equities: Invesco MSCI EM Universal Screened vs ACWI EM portion

The TKF model portfolio holds EM exposure (10%) through Invesco MSCI EM Universal Screened — which tracks the MSCI EM Universal ESG index, a different variant from the standard MSCI EM index embedded in ACWI. The "Universal" methodology tilts weights toward ESG leaders rather than simply excluding stocks, and applies different screening criteria.

**Data caveat:** Invesco holdings were scraped from companiesmarketcap.com, covering ~93% of the fund's weight. The ~7% missing weight inflates the active share figure — the true value is likely 3–5 pp lower.

In [9]:
# EM equities: Invesco MSCI EM Universal Screened vs ACWI EM portion
# Both have ISINs — direct join

# SPDR country names that map to EM
SPDR_EM_COUNTRIES = {
    'China', 'Taiwan', 'India', 'South Korea', 'Brazil', 'South Africa',
    'Mexico', 'Saudi Arabia', 'Thailand', 'Indonesia', 'Malaysia', 'Philippines',
    'Turkey', 'Poland', 'Chile', 'Qatar', 'United Arab Emirates', 'Kuwait',
    'Colombia', 'Peru', 'Czech Republic', 'Egypt', 'Greece', 'Hungary',
    'Hong Kong',  # Some EM stocks listed via HK
}

acwi_em = acwi_portfolio[acwi_portfolio['country'].isin(SPDR_EM_COUNTRIES)].copy()
acwi_em_total = acwi_em['weight'].sum()
acwi_em['weight'] = acwi_em['weight'] / acwi_em_total * 100
acwi_em['weight'] = acwi_em['weight'] / acwi_em['weight'].sum()  # normalize to 1.0

inv_em = all_etf_holdings['companiesmarketcap'].copy()
inv_em = inv_em[inv_em['isin'].astype(str).str.match(ISIN_RE, na=False)]
inv_em_agg = inv_em.groupby('isin').agg({'name': 'first', 'weight': 'sum'}).reset_index()
inv_em_agg['weight'] = inv_em_agg['weight'] / inv_em_agg['weight'].sum()

merged_em = pd.merge(inv_em_agg[['isin', 'name', 'weight']],
                     acwi_em[['isin', 'weight']].rename(columns={'weight': 'weight_acwi'}),
                     on='isin', how='outer')
merged_em['weight'] = merged_em['weight'].fillna(0)
merged_em['weight_acwi'] = merged_em['weight_acwi'].fillna(0)
merged_em['abs_diff'] = (merged_em['weight'] - merged_em['weight_acwi']).abs()
merged_em['diff'] = merged_em['weight'] - merged_em['weight_acwi']
merged_em['name'] = merged_em['name'].fillna('')
for idx, r in merged_em[merged_em['name'] == ''].iterrows():
    match = acwi_em[acwi_em['isin'] == r['isin']]
    if len(match): merged_em.loc[idx, 'name'] = match.iloc[0]['name']

em_active_share = merged_em['abs_diff'].sum() / 2
overlap_n = len(set(inv_em_agg['isin']) & set(acwi_em['isin']))

display(metric_box('EM: Invesco vs ACWI EM', f'{em_active_share:.2%}',
                   f'{len(inv_em_agg)} Invesco stocks vs {len(acwi_em)} ACWI EM stocks · {overlap_n} overlapping · '
                   f'Inflated ~3–5pp by incomplete Invesco data (93% coverage)'))
display(diff_table(merged_em, 'weight', 'weight_acwi', title='Top weight differences (EM)'))

,Stock,Fund %,Bench %,Diff,
1,TAIWAN SEMICONDUCTOR MANUFAC TWD10,6.15%,13.37%,-7.22%,UW
2,Samsung Electronics Co. Ltd.,0.00%,6.29%,-6.29%,UW
3,SK HYNIX INC KRW5000,5.24%,3.14%,+2.10%,OW
4,AIA Group Limited,0.00%,1.04%,-1.04%,UW
5,SAMSUNG ELECTRONICS-PREF KRW5000 PFD,0.86%,0.00%,+0.86%,OW
6,CHINA CONSTRUCTION BANK-H CNY1,1.74%,0.96%,+0.78%,OW
7,PDD Holdings Inc. Sponsored ADR Class A,0.00%,0.71%,-0.71%,UW
8,HDFC BANK LIMITED INR1,1.71%,1.06%,+0.65%,OW
9,Hong Kong Exchanges & Clearing Ltd.,0.00%,0.63%,-0.63%,UW
10,Vale S.A.,0.00%,0.57%,-0.57%,UW


### Europe, Japan, Pacific ex Japan: BNP Paribas Min TE vs iShares SAWD

The TKF model portfolio holds developed ex-North America exposure through three BNP Paribas Minimum Tracking Error ETFs: Europe (12.5%), Japan (3.7%), and Pacific ex Japan (1.9%). Unlike the other TKF funds, BNP Min TE ETFs do **not** apply ESG screening — they track unscreened MSCI regional indices with a minimum tracking error mandate.

We compare against the ex-North America portion of iShares MSCI World ESG Screened (SAWD). The high active share here largely reflects the **ESG screening gap**: BNP holds stocks like Nestle, Shell, and Unilever that SAWD excludes.

**Data caveat:** BNP Japan data covers ~96% of holdings. Some large stocks (e.g. Toyota Motor) may be in the unreported 4%.

In [10]:
# BNP Europe + Japan + Pacific ex Japan vs SAWD ex North America
# BNP has ISINs, SAWD bridged via SPDR

BNP_WEIGHTS = {'bnp_europe': 12.5, 'bnp_japan': 3.7, 'bnp_pacific': 1.9}
bnp_total_weight = sum(BNP_WEIGHTS.values())

bnp_stocks = []
for source, tkf_w in BNP_WEIGHTS.items():
    df = all_etf_holdings[source].copy()
    rel_w = tkf_w / bnp_total_weight
    for _, h in df.iterrows():
        isin = str(h.get('isin', '')) if pd.notna(h.get('isin', '')) else ''
        if ISIN_RE.match(isin):
            bnp_stocks.append({'isin': isin, 'name': h['name'], 'weight': rel_w * h['weight']})

bnp = pd.DataFrame(bnp_stocks)
bnp = bnp.groupby('isin').agg({'name': 'first', 'weight': 'sum'}).reset_index()
bnp['weight'] = bnp['weight'] / bnp['weight'].sum()

# SAWD ex-North America, bridged to ISINs
NA_COUNTRIES = {'United States', 'Canada'}
sawd_exna = sawd_with_isin[~sawd_with_isin['country'].isin(NA_COUNTRIES)].copy()
sawd_exna = sawd_exna[sawd_exna['isin'].notna()]
sawd_exna_total = sawd_exna['weight'].sum()
sawd_exna['weight'] = sawd_exna['weight'] / sawd_exna_total

print(f'BNP: {len(bnp)} stocks, SAWD ex-NA: {len(sawd_exna)} stocks (bridged to ISINs)')

merged_bnp = pd.merge(bnp[['isin', 'name', 'weight']],
                      sawd_exna[['isin', 'weight']].rename(columns={'weight': 'weight_sawd'}),
                      on='isin', how='outer')
merged_bnp['weight'] = merged_bnp['weight'].fillna(0)
merged_bnp['weight_sawd'] = merged_bnp['weight_sawd'].fillna(0)
merged_bnp['abs_diff'] = (merged_bnp['weight'] - merged_bnp['weight_sawd']).abs()
merged_bnp['diff'] = merged_bnp['weight'] - merged_bnp['weight_sawd']
merged_bnp['name'] = merged_bnp['name'].fillna('')
for idx, r in merged_bnp[merged_bnp['name'] == ''].iterrows():
    match = sawd_exna[sawd_exna['isin'] == r['isin']]
    if len(match): merged_bnp.loc[idx, 'name'] = match.iloc[0]['name']

bnp_active_share = merged_bnp['abs_diff'].sum() / 2
overlap_n = len(set(bnp['isin']) & set(sawd_exna['isin']))
bnp_only_n = len(set(bnp['isin']) - set(sawd_exna['isin']))

display(metric_box('DM ex-NA: BNP Min TE vs SAWD ex-NA', f'{bnp_active_share:.2%}',
                   f'{len(bnp)} BNP stocks vs {len(sawd_exna)} SAWD ex-NA stocks · {overlap_n} overlapping · '
                   f'{bnp_only_n} BNP-only'))
display(diff_table(merged_bnp, 'weight', 'weight_sawd', title='Top weight differences (DM ex-NA)'))

BNP: 609 stocks, SAWD ex-NA: 608 stocks (bridged to ISINs)


,Stock,Fund %,Bench %,Diff,
1,ROCHE HOLDING AG,0.07%,1.67%,-1.60%,UW
2,ROCHE HOLDING PAR AG,1.57%,0.00%,+1.57%,OW
3,NESTLE SA,1.36%,0.00%,+1.36%,OW
4,SIEMENS N AG,0.00%,1.12%,-1.12%,UW
5,TOYOTA MOTOR CORP,0.00%,1.12%,-1.12%,UW
6,SHELL PLC,1.08%,0.00%,+1.08%,OW
7,SIEMENS N AG,1.05%,0.00%,+1.05%,OW
8,UNILEVER PLC,0.76%,0.00%,+0.76%,OW
9,SIEMENS ENERGY N AG,0.66%,0.00%,+0.66%,OW
10,AIRBUS,0.57%,0.00%,+0.57%,OW


### Different index: Vanguard ESG North America All Cap

Vanguard ESG North America All Cap (15.4% of TKF) tracks the FTSE North America All Cap Choice Index — a different index family from the MSCI-based ETFs used by the other 8 funds. This creates active share through two channels:

1. **Small-cap stocks** — Vanguard's all-cap mandate includes ~1,000 small caps not in MSCI ACWI (large+mid cap only)
2. **Different ESG screening** — FTSE ESG excludes different companies than MSCI ESG Screened

We compare Vanguard vs Xtrackers USA + Canada (both held in TKF) to isolate what the Vanguard allocation adds beyond the MSCI-based funds.

**Data caveat:** Vanguard holdings are from 31 Jan 2026 (stale by ~6 weeks). The weight differences on overlapping stocks are partly due to price movements since then.

In [11]:
# Vanguard ESG NA All Cap vs Xtrackers USA + Canada
# Xtrackers has ISINs, Vanguard bridged via SPDR

acwi_us_weight = acwi_portfolio[acwi_portfolio['country'] == 'United States']['weight'].sum()
acwi_ca_weight = acwi_portfolio[acwi_portfolio['country'] == 'Canada']['weight'].sum()
us_share = acwi_us_weight / (acwi_us_weight + acwi_ca_weight)
ca_share = 1 - us_share

xt_usa = all_etf_holdings['xtrackers_usa'].copy()
xt_can = all_etf_holdings['xtrackers_canada'].copy()
xt_usa['weight_combined'] = xt_usa['weight'] * us_share
xt_can['weight_combined'] = xt_can['weight'] * ca_share

xt_combined = pd.concat([
    xt_usa[['isin', 'name', 'weight_combined']],
    xt_can[['isin', 'name', 'weight_combined']],
])
xt_combined = xt_combined.groupby('isin').agg({'name': 'first', 'weight_combined': 'sum'}).reset_index()
xt_combined['weight'] = xt_combined['weight_combined'] / xt_combined['weight_combined'].sum()

# Bridge Vanguard tickers to ISINs
vg = all_etf_holdings['vanguard_na'].copy()
vg['isin'] = vg.apply(lambda r: bridge_to_isin(r.get('ticker', ''), r['name'], r.get('country', '')), axis=1)
vg_matched = vg[vg['isin'].notna()].copy()
vg_unmatched = vg[vg['isin'].isna()]
vg_portfolio = vg_matched.groupby('isin').agg({'name': 'first', 'weight': 'sum'}).reset_index()
vg_portfolio['weight'] = vg_portfolio['weight'] / vg_portfolio['weight'].sum()

print(f'Vanguard: {len(vg)} stocks, {len(vg_matched)} bridged to ISINs, {len(vg_unmatched)} unmatched (small caps)')

xt_isins = set(xt_combined['isin'])
vg_isins = set(vg_portfolio['isin'])
vg_only = vg_isins - xt_isins
vg_only_weight = vg_portfolio[vg_portfolio['isin'].isin(vg_only)]['weight'].sum()

merged_vx = pd.merge(
    vg_portfolio[vg_portfolio['isin'].isin(xt_isins & vg_isins)][['isin', 'name', 'weight']],
    xt_combined[xt_combined['isin'].isin(xt_isins & vg_isins)][['isin', 'weight']].rename(columns={'weight': 'weight_xt'}),
    on='isin',
)
merged_vx['abs_diff'] = (merged_vx['weight'] - merged_vx['weight_xt']).abs()
merged_vx['diff'] = merged_vx['weight'] - merged_vx['weight_xt']
stale_active_share = merged_vx['abs_diff'].sum() / 2

display(metric_box('Vanguard ESG NA All Cap vs Xtrackers USA+Canada', '',
                   f'{len(vg_portfolio):,} Vanguard stocks vs {len(xt_combined)} Xtrackers · '
                   f'{len(xt_isins & vg_isins)} overlapping · '
                   f'{len(vg_only):,} Vanguard-only ({vg_only_weight:.2%} — mostly small caps) · '
                   f'Overlapping stocks weight diff (stale prices): {stale_active_share:.2%}'))
display(diff_table(merged_vx, 'weight', 'weight_xt', title='Top weight differences on overlapping stocks (Vanguard vs Xtrackers)'))

Vanguard: 1415 stocks, 641 bridged to ISINs, 774 unmatched (small caps)


,Stock,Fund %,Bench %,Diff,
1,Alphabet Inc,6.91%,3.10%,+3.81%,OW
2,Microsoft Corp,6.18%,5.03%,+1.15%,OW
3,John Bean Technologies Corp,0.02%,1.02%,-1.00%,UW
4,NVIDIA Corp,8.67%,7.73%,+0.94%,OW
5,Amazon.com Inc,4.45%,3.58%,+0.87%,OW
6,Apple Inc,7.33%,6.65%,+0.68%,OW
7,Meta Platforms Inc,3.03%,2.46%,+0.58%,OW
8,Tesla Inc,2.35%,1.96%,+0.39%,OW
9,InterDigital Inc,0.03%,0.41%,-0.38%,UW
10,Nextpower Inc,0.03%,0.33%,-0.30%,UW
